# Proyecto Final

In [36]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("Pract2_PySpark") \
    .getOrCreate()

# Verificar la versión
print("Versión de Spark:", spark.version)

sc = spark.sparkContext

DATA_PATH = "/home/FinPlus/work/data/"

Versión de Spark: 3.5.0


## Impotamos la data

In [37]:
client = (spark.read.option('header', 'true').option('delimiter',',')
                     .csv(DATA_PATH + 'CLIENTS.csv'))
client.show(5, 0)

+------------+----------------------+-----------------+------+------------+--------------+-----------+--------------+--------------+--------------------+------------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|CLIENT_ID   |NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION     |MARITAL_STATUS|HOME_SITUATION      |REGION_SC

In [ ]:
client.printSchema()

In [ ]:
client.columns

In [47]:
beh = (spark.read.parquet(DATA_PATH + 'BEHAVIOURAL_PARQUET'))
beh = beh.withColumnRenamed('CURRENCY', 'B_CURRENCY')
beh.show(5, 0)

+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+----------+
|CONTRACT_ID       |CLIENT_ID   |DATE      |CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|B_CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+----------+
|ES1821190439i00XXX|ES182363269V|2020-08-22|4189.73             |5400.0           |1193.4                  |1193.4              |0.0                     |0.0                       |162.0          

In [ ]:
beh.printSchema()

In [ ]:
beh.columns

# Quitamos los duplicados en los dos data sets

In [39]:
client_sin_duplicados_por_id = client.dropDuplicates(['CLIENT_ID'])

client_sin_duplicados_por_id.show(5)

print(f"Filas originales en client: {client.count()}")
print(f"Filas después de eliminar duplicados (por CLIENT_ID) en client: {client_sin_duplicados_por_id.count()}")

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+----------------+------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|   CLIENT_ID|NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION|MARITAL_STATUS|  HOME_SITUATION|REGION_SCORE|      AGE_IN_YEARS|J

In [40]:
beh_sin_duplicados_por_id = beh.dropDuplicates(['CLIENT_ID', 'DATE', 'CREDICT_CARD_BALANCE', 'CONTRACT_ID'])

beh_sin_duplicados_por_id.show(5)

print(f"Filas originales en beh: {beh.count()}")
print(f"Filas después de eliminar duplicados (por CLIENT_ID) en beh: {beh_sin_duplicados_por_id.count()}")

+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|       CONTRACT_ID|   CLIENT_ID|      DATE|CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|ES1821489396v00XXX|ES182100006A|2021-07-29|                 0.0|           3240.0|                     0.0|                 0.0|                     0.0|                       0.0|                0.0| 

# Contamos el número de nulls que hay en cada una de las columnas

In [ ]:
for column in client.columns:
    null_count = client.filter(client[column].isNull()).count()
    print(f"Columna '{column}': {null_count} valores nulos")

In [ ]:
for column in beh.columns:
    null_count = beh.filter(beh[column].isNull()).count()
    print(f"Columna '{column}': {null_count} valores nulos")

# Cambiamos Nulls

In [41]:
df_client = client

score_columns = ['REACTIVE_SCORING', 'PROACTIVE_SCORING', 'BEHAVIORAL_SCORING']

for col_name in score_columns:
    df_client = df_client.withColumn(col_name, F.col(col_name).cast(T.FloatType()))

avg_reactive_scoring = df_client.filter(F.col('REACTIVE_SCORING').isNotNull()).agg(F.avg('REACTIVE_SCORING')).collect()[0][0]
avg_proactive_scoring = df_client.filter(F.col('PROACTIVE_SCORING').isNotNull()).agg(F.avg('PROACTIVE_SCORING')).collect()[0][0]
avg_behavioral_scoring = df_client.filter(F.col('BEHAVIORAL_SCORING').isNotNull()).agg(F.avg('BEHAVIORAL_SCORING')).collect()[0][0]

fill_values = {
    'INSTALLMENT': 0.0,
    'EDUCATION': 'Bachelor',
    'MARITAL_STATUS': 'NA',
    'JOB_SENIORITY': 0.0,
    'CAR_AGE': 0.0,
    'FAMILY_SIZE': 1.0,
    'REACTIVE_SCORING': avg_reactive_scoring,
    'PROACTIVE_SCORING': avg_proactive_scoring,
    'BEHAVIORAL_SCORING': avg_behavioral_scoring,
    'DAYS_LAST_INFO_CHANGE': 0.0,
    'NUMBER_OF_PRODUCTS': 0.0,
    'EMPLOYER_ORGANIZATION_TYPE': 'NA',
    'NUM_PREVIOUS_LOAN_APP': 0.0,
    'LOAN_ANNUITY_PAYMENT_MAX': 0.0,
    'LOAN_ANNUITY_PAYMENT_MIN': 0.0,
    'LOAN_ANNUITY_PAYMENT_SUM': 0.0,
    'LOAN_APPLICATION_AMOUNT_MAX': 0.0,
    'LOAN_APPLICATION_AMOUNT_MIN': 0.0,
    'LOAN_APPLICATION_AMOUNT_SUM': 0.0,
    'LOAN_CREDIT_GRANTED_MAX': 0.0,
    'LOAN_CREDIT_GRANTED_MIN': 0.0,
    'LOAN_CREDIT_GRANTED_SUM': 0.0,
    'LOAN_VARIABLE_RATE_MAX': 0.0,
    'LOAN_VARIABLE_RATE_MIN': 0.0,
    'NUM_STATUS_ANNULLED': 0.0,
    'NUM_STATUS_AUTHORIZED': 0.0,
    'NUM_STATUS_DENIED': 0.0,
    'NUM_STATUS_NOT_USED': 0.0,
    'NUM_FLAG_INSURED': 0.0
}

df_client = df_client.na.fill(fill_values)

print("Client DataFrame after handling missing values. Verifying null counts:")
for column in df_client.columns:
    null_count = df_client.filter(F.col(column).isNull()).count()
    if null_count > 0:
        print(f"Columna '{column}': {null_count} valores nulos")
    else:
        print(f"Columna '{column}': 0 valores nulos")

df_client.show(5)

Client DataFrame after handling missing values. Verifying null counts:
Columna 'CLIENT_ID': 0 valores nulos
Columna 'CLIENT_ID': 0 valores nulos
Columna 'NON_COMPLIANT_CONTRACT': 0 valores nulos
Columna 'NON_COMPLIANT_CONTRACT': 0 valores nulos
Columna 'NAME_PRODUCT_TYPE': 0 valores nulos
Columna 'NAME_PRODUCT_TYPE': 0 valores nulos
Columna 'GENDER': 0 valores nulos
Columna 'GENDER': 0 valores nulos
Columna 'TOTAL_INCOME': 0 valores nulos
Columna 'TOTAL_INCOME': 0 valores nulos
Columna 'AMOUNT_PRODUCT': 0 valores nulos
Columna 'INSTALLMENT': 0 valores nulos
Columna 'EDUCATION': 0 valores nulos
Columna 'MARITAL_STATUS': 0 valores nulos
Columna 'AMOUNT_PRODUCT': 0 valores nulos
Columna 'INSTALLMENT': 0 valores nulos
Columna 'EDUCATION': 0 valores nulos
Columna 'MARITAL_STATUS': 0 valores nulos
Columna 'HOME_SITUATION': 0 valores nulos
Columna 'HOME_SITUATION': 0 valores nulos
Columna 'REGION_SCORE': 0 valores nulos
Columna 'REGION_SCORE': 0 valores nulos
Columna 'AGE_IN_YEARS': 0 valores

# Union de datasets y cambio de tipo de datos

In [48]:
# Definir una ventana particionada por cliente y ordenada por fecha descendente
ventana = Window.partitionBy("CLIENT_ID").orderBy(F.col("DATE").desc())

# Añadir un ranking a las transacciones (1 = la más reciente)
df_beh_ranking = beh.withColumn("rank", F.row_number().over(ventana))

# Filtrar solo la última transacción
df_best = df_beh_ranking.filter(F.col("rank") == 1).drop("rank")

# Unir con clientes
df_clean = df_client.join(df_best, on="CLIENT_ID",how = "left")

df_clean.show()


+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------------+------------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+-----------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----

In [53]:
nulos = {
    'CREDICT_CARD_BALANCE': 0.0,
    'CONTRACT_ID': 'NA',
    'DATE' : 'NA',
    'CREDIT_CARD_LIMIT': 0.0,
    'CREDIT_CARD_DRAWINGS_ATM': 0.0,
    'CREDIT_CARD_DRAWINGS':0.0,
    'CREDIT_CARD_DRAWINGS_OTHER': 0.0,
    'CREDIT_CARD_DRAWINGS_POS': 0.0,
    'CREDIT_CARD_PAYMENT': 0.0,
    'NUMBER_DRAWINGS_ATM': 0.0,
    'NUMBER_DRAWINGS': 0.0,
    'NUMBER_INSTALMENTS': 0.0,
    'B_CURRENCY': 'euros'
}

df_clean = df_clean.na.fill(nulos)

df_clean.show()

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------------+------------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+-----------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----

In [54]:
(df_clean.groupBy('CREDICT_CARD_BALANCE')
    .count()
    .orderBy(F.desc('count'))
).show(10)

+--------------------+------+
|CREDICT_CARD_BALANCE| count|
+--------------------+------+
|                 0.0|146649|
|                1.57|    31|
|                3.24|    18|
|              558.85|    10|
|                0.01|     8|
|                0.81|     8|
|                2.38|     7|
|               18.85|     7|
|                1.54|     6|
|                1.62|     5|
+--------------------+------+
only showing top 10 rows



In [55]:
NewDataTypes = ['NON_COMPLIANT_CONTRACT',
 'TOTAL_INCOME',
 'AMOUNT_PRODUCT',
 'INSTALLMENT',
 'REGION_SCORE',
 'AGE_IN_YEARS',
 'JOB_SENIORITY',
 'HOME_SENIORITY',
 'LAST_UPDATE',
 'CAR_AGE',
 'FAMILY_SIZE',
 'REACTIVE_SCORING',
 'PROACTIVE_SCORING',
 'BEHAVIORAL_SCORING',
 'DAYS_LAST_INFO_CHANGE',
 'NUMBER_OF_PRODUCTS',
 'DIGITAL_CLIENT',
 'EMPLOYER_ORGANIZATION_TYPE',
 'NUM_PREVIOUS_LOAN_APP',
 'LOAN_ANNUITY_PAYMENT_MAX',
 'LOAN_ANNUITY_PAYMENT_MIN',
 'LOAN_ANNUITY_PAYMENT_SUM',
 'LOAN_APPLICATION_AMOUNT_MAX',
 'LOAN_APPLICATION_AMOUNT_MIN',
 'LOAN_APPLICATION_AMOUNT_SUM',
 'LOAN_CREDIT_GRANTED_MAX',
 'LOAN_CREDIT_GRANTED_MIN',
 'LOAN_CREDIT_GRANTED_SUM',
 'LOAN_VARIABLE_RATE_MAX',
 'LOAN_VARIABLE_RATE_MIN',
 'NUM_STATUS_ANNULLED',
 'NUM_STATUS_AUTHORIZED',
 'NUM_STATUS_DENIED',
 'NUM_STATUS_NOT_USED',
 'NUM_FLAG_INSURED',
 'CREDICT_CARD_BALANCE',
 'CREDIT_CARD_LIMIT',
 'CREDIT_CARD_DRAWINGS_ATM',
 'CREDIT_CARD_DRAWINGS',
 'CREDIT_CARD_DRAWINGS_POS',
 'CREDIT_CARD_DRAWINGS_OTHER',
 'CREDIT_CARD_PAYMENT',
 'NUMBER_DRAWINGS_ATM',
 'NUMBER_DRAWINGS',
 'NUMBER_INSTALMENTS']

for i in NewDataTypes:
    df_clean = df_clean.withColumn(i, F.col(i).cast('float'))

df_clean.printSchema()

root
 |-- CLIENT_ID: string (nullable = true)
 |-- NON_COMPLIANT_CONTRACT: float (nullable = true)
 |-- NAME_PRODUCT_TYPE: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- TOTAL_INCOME: float (nullable = true)
 |-- AMOUNT_PRODUCT: float (nullable = true)
 |-- INSTALLMENT: float (nullable = true)
 |-- EDUCATION: string (nullable = false)
 |-- MARITAL_STATUS: string (nullable = false)
 |-- HOME_SITUATION: string (nullable = true)
 |-- REGION_SCORE: float (nullable = true)
 |-- AGE_IN_YEARS: float (nullable = true)
 |-- JOB_SENIORITY: float (nullable = true)
 |-- HOME_SENIORITY: float (nullable = true)
 |-- LAST_UPDATE: float (nullable = true)
 |-- OWN_INSURANCE_CAR: string (nullable = true)
 |-- CAR_AGE: float (nullable = true)
 |-- FAMILY_SIZE: float (nullable = true)
 |-- REACTIVE_SCORING: float (nullable = false)
 |-- PROACTIVE_SCORING: float (nullable = false)
 |-- BEHAVIORAL_SCORING: float (nullable = false)
 |-- DAYS_LAST_INFO_CHANGE: float (nullable = true)
 |--

# Transformacion de la data

In [ ]:
(df_client.groupBy('NAME_PRODUCT_TYPE')
    .count()
    .orderBy(F.desc('count'))
).show(10)

+-----------------+------+
|NAME_PRODUCT_TYPE| count|
+-----------------+------+
|        PRODUCT 1|147470|
|        PRODUCT 2| 15507|
+-----------------+------+



In [9]:
# A PARTIR DE AHORA SON LAS METRICAS SIN ARREGLAR

In [27]:
(df_clean.groupBy(
    'DIGITAL_CLIENT').count().orderBy(F.desc('count'))
).show(10)

+--------------+------+
|DIGITAL_CLIENT| count|
+--------------+------+
|           0.0|153832|
|           1.0|  9145|
+--------------+------+



In [28]:
(df_clean.filter(df_clean['DIGITAL_CLIENT'] == '1').count())/(df_clean.count())*100


5.6112212152635035

In [ ]:
df_clean = df_clean.na.fill({'CREDICT_CARD_BALANCE':0.0})

df = df_clean.select(
    F.col('INSTALLMENT'),
    F.col('CREDICT_CARD_BALANCE'),
    F.col('TOTAL_INCOME'),
    (((F.col('INSTALLMENT') + (F.col('CREDICT_CARD_BALANCE') * 0.05)) / F.col('TOTAL_INCOME')) * 100).alias('DEBT_RATIO')
).withColumn(
    "Riesgo",
    F.when(F.col("DEBT_RATIO") > 45, "Alto Riesgo")
    .otherwise("Bajo Riesgo")
)

df.show()

(df.groupBy(
    'Riesgo').count().orderBy(F.desc('count'))
).show(10)


In [ ]:
# Agrupar y sumar
df_totals = df_clean.groupBy("CLIENT_ID").agg(
    F.sum("CREDIT_CARD_DRAWINGS_ATM").alias("total_atm"),
    F.sum("CREDIT_CARD_DRAWINGS_POS").alias("total_pos"),
    F.sum("CREDIT_CARD_DRAWINGS").alias("total_gastos")
)

# Calcular porcentajes
df_percentages = df_totals.withColumn(
    "pct_atm",
    (F.col("total_atm") / F.col("total_gastos")) * 100
).withColumn(
    "pct_pos",
    (F.col("total_pos") / F.col("total_gastos")) * 100
).fillna(0)

# Segmentar
df_segmented = df_percentages.withColumn(
    "segmento",
    F.when(F.col("pct_atm") > 70, "Efectivo-dependiente")
    .when(F.col("pct_pos") > 70, "Cashless")
    .otherwise("Mixto")
)

# Mostrar resultados
df_segmented.show()

(df_segmented.groupBy(
    'segmento').count().orderBy(F.desc('count'))
).show(10)

In [70]:
#este iria en oportunidad comercial

df_cazadores = df_clean.filter(df_clean['NUM_STATUS_DENIED'] > 0)

# --- 1. Definición de Percentiles a Usar ---
# Para TOTAL_INCOME (Ingreso Alto): Usamos el Percentil 75 (0.75)
PROB_INGRESO = 0.75 

# Para BEHAVIORAL_SCORING (Buen Comportamiento): Usamos el Percentil 60 (0.60)
PROB_SCORE = 0.60

# --- 2. Cálculo del Umbral de Ingreso Alto (Q3) ---

# approxQuantile devuelve una lista, extraemos el primer (y único) valor [0]
# El tercer argumento (0.01) es la precisión; no lo cambies.
umbral_ingreso_alto = df_clean.stat.approxQuantile(
    "TOTAL_INCOME", 
    [PROB_INGRESO], 
    0.01
)[0]

# --- 3. Cálculo del Umbral de Behavioral Scoring (P60) ---

umbral_score_alto = df_clean.stat.approxQuantile(
    "BEHAVIORAL_SCORING", 
    [PROB_SCORE], 
    0.01
)[0]

df_cazadores_filtrado = df_cazadores.filter(
    (df_cazadores['TOTAL_INCOME'] >= umbral_ingreso_alto) & 
    (df_cazadores['BEHAVIORAL_SCORING'] >= umbral_score_alto)
)

# Calcula la Tasa de Utilización (CCU)
df_cazadores_final = df_cazadores_filtrado.withColumn(
    "CC_UTILIZATION_RATE", 
    F.col("CREDICT_CARD_BALANCE") / F.col("CREDIT_CARD_LIMIT")
)

# Asigna un "Puntaje de Oportunidad"
# Los que tienen bajo uso de CC son mejores candidatos
df_cazadores_final = df_cazadores_final.withColumn(
    "OPORTUNIDAD_SCORE",
    F.col("BEHAVIORAL_SCORING") * (1 - F.col("CC_UTILIZATION_RATE"))
)

# Ordenar para priorizar los mejores candidatos
df_cazadores_final = df_cazadores_final.orderBy(F.col("OPORTUNIDAD_SCORE").desc())

# Muestra los 20 mejores candidatos para una oferta pre-concedida
df_cazadores_final.show(20)

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------------+------------+------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+-----------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----------------

In [76]:
# Paso 1: Crear rangos de Seniority en el DataFrame 'clients'
df_seniority_grouped = df_clean.select('CLIENT_ID', 'JOB_SENIORITY', 'PROACTIVE_SCORING').withColumn(
    "SENIORITY_GROUP",
    F.when(F.col("JOB_SENIORITY") <= 2, "Baja Seniority (0-2 años)")
    .when((F.col("JOB_SENIORITY") > 2) & (F.col("JOB_SENIORITY") <= 5), "Media Seniority (2-5 años)")
    .otherwise("Alta Seniority (> 5 años)")
)

# Paso 2: Agrupar por rango de Seniority y calcular el score promedio
df_analisis_estabilidad = df_seniority_grouped.groupBy("SENIORITY_GROUP").agg(
    F.avg("PROACTIVE_SCORING").alias("PROMEDIO_SCORE"),
    F.count("CLIENT_ID").alias("NUM_CLIENTES_GRUPO")
)

# Paso 3: Mostrar el resultado ordenado
df_analisis_estabilidad.orderBy(F.col("PROMEDIO_SCORE").desc()).show()

+--------------------+------------------+------------------+
|     SENIORITY_GROUP|    PROMEDIO_SCORE|NUM_CLIENTES_GRUPO|
+--------------------+------------------+------------------+
|Media Seniority (...|0.5928756952285766|                 5|
|Alta Seniority (>...|0.5157750902221804|            133797|
|Baja Seniority (0...|0.5063301750825089|             29175|
+--------------------+------------------+------------------+



In [ ]:
# Etiquetamos al cliente: ¿Es de Ciudad o de Pueblo?
df = df_clean.withColumn(
    "ZONA",
    F.when(F.col("REGION_SCORE") >= 3, "Urbana")
     .otherwise("Rural")
)

# Agrupamos y calculamos lo importante en un solo paso
df_analisis = df.groupBy("ZONA").agg(
    F.count("CLIENT_ID").alias("Num_Clientes"),
    F.round(F.avg("TOTAL_INCOME"), 0).alias("Sueldo_Promedio"),
    F.sum("NUM_STATUS_DENIED").alias("Total_Rechazos")
)

# Calculamos el % de rechazo final para comparar
df_final = df_analisis.withColumn(
    "Indice_Rechazo", 
    F.round((F.col("Total_Rechazos") / F.col("Num_Clientes")) * 100, 2)
)

df_final.show()

# Actividad del cliente

In [ ]:
(df_clean.groupBy('DAYS_LAST_INFO_CHANGE')
    .count()
    .orderBy(F.desc('count'))
).show(10)

# Valor Económico

In [ ]:
# Ver si un cliente esta 100% con nosotros o si esta en otros bancos y si es rentable traerlo

df_sow_analysis = df_clean.withColumn(
    "TOTAL_PAYMENTS_TO_BANK", 
    F.col("INSTALLMENT") + F.col("AVG_CC_PAYMENT")
).withColumn(
    "SHARE_OF_WALLET",
    F.when(F.col("TOTAL_INCOME") > 0, 
           F.col("TOTAL_PAYMENTS_TO_BANK") / F.col("TOTAL_INCOME")
    ).otherwise(0) # Si no declara ingresos, ponemos 0 para evitar división por cero
)

df_final_strategy = df_sow_analysis.withColumn(
    "ESTRATEGIA_COMERCIAL",
    F.when(F.col("SHARE_OF_WALLET") <= 0.10, "Ataque (Traer Nómina/Hipoteca)")
     .when(F.col("SHARE_OF_WALLET").between(0.10, 0.40), "Crecimiento (Cross-Sell)")
     .when(F.col("SHARE_OF_WALLET").between(0.40, 0.60), "Fidelización (Blindaje)")
     .when(F.col("SHARE_OF_WALLET") > 0.60, "Alerta Riesgo (Posible Impago)")
     .otherwise("Revisar Datos")
)

df_final_strategy.select(
    "CLIENT_ID", "TOTAL_INCOME", "TOTAL_PAYMENTS_TO_BANK", "SHARE_OF_WALLET", "ESTRATEGIA_COMERCIAL"
).show(10)

In [ ]:
# Ratio de conversión de credito
df_clean.select(
    F.col('LOAN_APPLICATION_AMOUNT_SUM'),
    F.col('LOAN_CREDIT_GRANTED_SUM')
).withColumn(
    'CREDIT_GARANTED_SCORE', F.col('LOAN_CREDIT_GRANTED_SUM') / F.col('LOAN_APPLICATION_AMOUNT_SUM')
).show()

# Interacción y Fidelidad

In [62]:
df_clean.select(F.mean("NUMBER_OF_PRODUCTS"), F.max("NUMBER_OF_PRODUCTS")).show()

df_cross_sell = df_clean.withColumn(
    "VINCULACION",
    F.when(F.col("NUMBER_OF_PRODUCTS") == 1, "Baja (Riesgo Fuga)")
     .when(F.col("NUMBER_OF_PRODUCTS").between(2, 3), "Media")
     .when(F.col("NUMBER_OF_PRODUCTS") >= 4, "Alta (Fidelizado)")
     .otherwise("Desconocido")
)

df_cross_sell.groupBy("VINCULACION").count().show()

+-----------------------+-----------------------+
|avg(NUMBER_OF_PRODUCTS)|max(NUMBER_OF_PRODUCTS)|
+-----------------------+-----------------------+
|      1.644523460365573|                   20.5|
+-----------------------+-----------------------+

+------------------+-----+
|       VINCULACION|count|
+------------------+-----+
|             Media|44458|
|Baja (Riesgo Fuga)|33676|
|       Desconocido|59984|
| Alta (Fidelizado)|24859|
+------------------+-----+



In [63]:
df_stickiness = df_clean.withColumn(
    "TOTAL_SOLICITUDES_HISTORICAS",
    F.col("NUM_STATUS_ANNULLED") + F.col("NUM_STATUS_AUTHORIZED") +
    F.col("NUM_STATUS_DENIED") + F.col("NUM_STATUS_NOT_USED")
).withColumn(
    "RATIO_VITRINEO", # % de veces que el cliente rechazó la oferta aprobada
    F.when(F.col("TOTAL_SOLICITUDES_HISTORICAS") > 0,
           F.col("NUM_STATUS_NOT_USED") / F.col("TOTAL_SOLICITUDES_HISTORICAS")
    ).otherwise(0)
)

# Análisis: Clientes que frecuentemente rechazan ofertas
df_stickiness.filter(F.col("RATIO_VITRINEO") > 0.5).select("CLIENT_ID", "RATIO_VITRINEO").show(5)

+------------+------------------+
|   CLIENT_ID|    RATIO_VITRINEO|
+------------+------------------+
|ES182201729A|0.6666666666666666|
|ES182241009M|               0.6|
|ES182287694L|              0.75|
|ES182253955P|0.6666666666666666|
|ES182158899Q|0.5454545454545454|
+------------+------------------+
only showing top 5 rows



In [ ]:
df_behavior_agg = df_clean.groupBy("CLIENT_ID").agg(
    F.sum("CREDIT_CARD_DRAWINGS_ATM").alias("TOTAL_ATM_AMOUNT"),
    F.sum("CREDIT_CARD_DRAWINGS_POS").alias("TOTAL_POS_AMOUNT"),
    F.sum("CREDIT_CARD_DRAWINGS").alias("TOTAL_DRAWINGS_ALL"))

# 3. Calculamos el KPI de "Digitalización Transaccional"
# Si TOTAL_DRAWINGS_ALL es 0, asumimos 0 digitalización para evitar error
df_digital_kpi = df_clean.withColumn(
    "RATIO_USO_DIGITAL", 
    F.when(F.col("TOTAL_DRAWINGS_ALL") > 0,
           F.col("TOTAL_POS_AMOUNT") / F.col("TOTAL_DRAWINGS_ALL")
    ).otherwise(0)
)

# 4. Matriz de Confusión de Digitalización
# Comparamos lo que el banco CREE (DIGITAL_CLIENT) con la REALIDAD (RATIO > 50%)
df_digital_kpi.withColumn(
    "PERFIL_REAL",
    F.when((F.col("DIGITAL_CLIENT") == 1) & (F.col("RATIO_USO_DIGITAL") > 0.5), "Digital Puro")
     .when((F.col("DIGITAL_CLIENT") == 1) & (F.col("RATIO_USO_DIGITAL") <= 0.5), "Digital de Fachada (Usa Cash)")
     .when((F.col("DIGITAL_CLIENT") == 0) & (F.col("RATIO_USO_DIGITAL") > 0.5), "Digital Potencial (Sin App)")
     .otherwise("Analógico")
).groupBy("PERFIL_REAL").count().show()

# Riesgo Potencial

In [66]:
df = df_clean.select(
    F.col('NUM_STATUS_DENIED'),
    F.col('NUM_STATUS_AUTHORIZED'),
    F.when(
        (F.col('NUM_STATUS_AUTHORIZED') + F.col('NUM_STATUS_DENIED')) == 0,
        F.lit(None) # Set to None (null) when denominator is zero
    ).otherwise(
        ((F.col('NUM_STATUS_DENIED') / (F.col('NUM_STATUS_AUTHORIZED') + F.col('NUM_STATUS_DENIED'))) * 100)
    ).alias('PCT_RECHAZO')
).withColumn(
    "Riesgo",
    F.when(F.col("PCT_RECHAZO").isNull(), "Indefinido")
    .when(F.col("PCT_RECHAZO") > 60, "Alto riesgo")
    .otherwise("Bajo Riesgo")
)
df.show()

(df.groupBy(
    'Riesgo').count().orderBy(F.desc('count'))
).show(10)

+-----------------+---------------------+----------------+-----------+
|NUM_STATUS_DENIED|NUM_STATUS_AUTHORIZED|     PCT_RECHAZO|     Riesgo|
+-----------------+---------------------+----------------+-----------+
|              0.0|                  5.0|             0.0|Bajo Riesgo|
|              0.0|                  1.0|             0.0|Bajo Riesgo|
|              0.0|                  3.0|             0.0|Bajo Riesgo|
|              0.0|                  2.0|             0.0|Bajo Riesgo|
|              0.0|                  1.0|             0.0|Bajo Riesgo|
|              0.0|                  0.0|            NULL| Indefinido|
|              0.0|                 10.0|             0.0|Bajo Riesgo|
|              0.0|                  1.0|             0.0|Bajo Riesgo|
|              0.0|                  2.0|             0.0|Bajo Riesgo|
|              0.0|                  2.0|             0.0|Bajo Riesgo|
|              0.0|                  2.0|             0.0|Bajo Riesgo|
|     

# OPORTUNIDADES COMERCIALES

In [64]:
df = df_clean
(df.groupBy('CAR_AGE', 'OWN_INSURANCE_CAR')
    .count()
    .orderBy(F.desc('count'))
).show()

+-------+-----------------+------+
|CAR_AGE|OWN_INSURANCE_CAR| count|
+-------+-----------------+------+
|    0.0|                N|107548|
|    7.0|                Y|  3949|
|    6.0|                Y|  3364|
|    3.0|                Y|  3345|
|    2.0|                Y|  3163|
|    8.0|                Y|  3123|
|    4.0|                Y|  2924|
|    1.0|                Y|  2776|
|    9.0|                Y|  2715|
|   10.0|                Y|  2543|
|   13.0|                Y|  2459|
|   14.0|                Y|  2404|
|   11.0|                Y|  2269|
|   12.0|                Y|  2261|
|    5.0|                Y|  1926|
|   15.0|                Y|  1860|
|   16.0|                Y|  1758|
|   17.0|                Y|  1527|
|   18.0|                Y|  1272|
|   64.0|                Y|  1256|
+-------+-----------------+------+
only showing top 20 rows



In [ ]:
df_coche = df_clean.filter(
    (F.col("OWN_INSURANCE_CAR") == 1) &       
    (F.col("CAR_AGE") >= 10) &                
    (F.col("TOTAL_INCOME") >= 2500)           
)

# Creamos una columna de "Prioridad" para que el Call Center llame primero a los mejores
df_priorizado = df_coche.withColumn(
    "PRIORIDAD_VENTA",
    F.when(F.col("CAR_AGE") > 15, "1. Urgente (Chatarra)")
     .when(F.col("CAR_AGE").between(10, 15), "2. Alta (Renovación)")
     .otherwise("3. Media")
)

# Ordenamos por ingresos (primero los que más pueden gastar)
df_recambio_coche = df_priorizado.select(
    F.col("CLIENT_ID"), 
    F.col("NAME_PRODUCT_TYPE"), 
    F.col("CAR_AGE"), 
    F.coll("TOTAL_INCOME"), 
    F.col("PRIORIDAD_VENTA")
).orderBy(F.col("TOTAL_INCOME").desc())

df_recambio_coche.show(10)

In [ ]:
# 4. Cálculo del Ratio de Utilización de Crédito (RUC) Mensual
# RUC = (CREDIT_CARD_BALANCE / CREDIT_CARD_LIMIT) * 100
df_ruc_mensual = df_clean.withColumn(
    "RUC_MENSUAL",
    # Manejo de divisiones por cero (límites nulos o cero)
    F.when(F.col("CREDIT_CARD_LIMIT") > 0, (F.col("CREDICT_CARD_BALANCE") / F.col("CREDIT_CARD_LIMIT")) * 100)
    .otherwise(F.lit(0.0))
)

# 5. Agregación por Cliente: Calcular el RUC Promedio
df_metrics_agg = df_ruc_mensual.groupBy("CLIENT_ID").agg(
    F.avg("RUC_MENSUAL").alias("AVG_RUC_PERCENT")
)

# 6. Clasificación y Conteo

# Criterio 1: Uso Extremo Alto (90% o más)
df_alto_uso = df_metrics_agg.filter(F.col("AVG_RUC_PERCENT") >= 90.0)
count_alto_uso = df_alto_uso.count()

# Criterio 2: Uso Extremo Bajo (10% o menos)
df_bajo_uso = df_metrics_agg.filter(F.col("AVG_RUC_PERCENT") <= 10.0)
count_bajo_uso = df_bajo_uso.count()

total_clientes = df_metrics_agg.count()

# 7. Presentación de Resultados
print("\n--- 📈 Resultados del Análisis de Utilización de Crédito ---")

print(f"Total de clientes analizados: {total_clientes}")
print("-" * 40)

# Resultado 1: Clientes con Alta Dependencia (Alto Riesgo/Wallet Share)
print(f"Cantidad de clientes que usan el 90% o más (RUC >= 90%):")
print(f"  Clientes: {count_alto_uso}")
if total_clientes > 0:
    percent_alto_uso = (count_alto_uso / total_clientes) * 100
    print(f"  Porcentaje: {percent_alto_uso:.2f}%")

# Resultado 2: Clientes con Baja Dependencia (Bajo Riesgo/Wallet Share)
print(f"\nCantidad de clientes que usan el 10% o menos (RUC <= 10%):")
print(f"  Clientes: {count_bajo_uso}")
if total_clientes > 0:
    percent_bajo_uso = (count_bajo_uso / total_clientes) * 100
    print(f"  Porcentaje: {percent_bajo_uso:.2f}%")
